PhylogeneticTreeGeneration

In [ ]:
from ete3 import Tree

# Load your Newick tree
tree = Tree("RaxML.nwk")

# List of bacteria to keep
keep = [
    "cxy", "cohm", "mip", "rfa", "bbet", "cauj", "dri", "mgk", "pmor",
    "tro", "cuvk", "cpi", "flk", "mlot", "buol", "lur", "lgu", "pzu", 
    "pscf", "racd", "vpe"
]

# Prune the tree to keep only those taxa
tree.prune(keep, preserve_branch_length=True)

# Write the pruned tree to a new file
tree.write(outfile="pruned_tree.nwk")


In [ ]:
from ete3 import Tree, TreeStyle

# Load your Newick tree
tree = Tree("RaxML.nwk")

# List of bacteria to keep
keep = [
    "cxy", "cohm", "mip", "rfa", "bbet", "cauj", "dri", "mgk", "pmor",
    "tro", "cuvk", "cpi", "flk", "mlot", "buol", "lur", "lgu", "pzu", 
    "pscf", "racd", "vpe"
]

# Prune the tree to keep only those taxa
tree.prune(keep, preserve_branch_length=True)

# Set up a TreeStyle object for circular visualization
ts = TreeStyle()
ts.mode = "c"  # Circular mode
ts.show_branch_length = True  # Optionally show branch lengths
ts.show_branch_support = True  # Optionally show branch support values

# Render the tree and save it as an image
tree.render("pruned_tree_circular.png", w=183, units="mm", tree_style=ts)

# Write the pruned tree to a new file
tree.write(outfile="pruned_tree.nwk")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from Bio import Phylo

# Load the Newick tree
tree_path = 'pruned_tree.nwk'  # Replace with your tree file path
tree = Phylo.read(tree_path, 'newick')

# Root the tree at its midpoint
tree.root_at_midpoint()

# Create a new figure for the dendrogram
plt.figure(figsize=(8, 12))

# Plot the rooted dendrogram from the phylogenetic tree
Phylo.draw(tree, do_show=False)

# Save the dendrogram as a PDF
dendrogram_pdf_path = "biolog_dendrogram_rooted.pdf"
plt.savefig(dendrogram_pdf_path, format="pdf", bbox_inches="tight")

print(f"Dendrogram saved as '{dendrogram_pdf_path}'")


CircularPlot

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib
# === Load Data ===
file_path = './GEMStats.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet2')
# === Define phyla mapping ===
phyla_dict = {
    "cohm": "Firmicutes",
    "cuvk": "Actinobacteria",
    "cxy": "Actinobacteria",
    "mip": "Actinobacteria",
    "rfa": "Actinobacteria",
    "cpi": "Bacteroidetes",
    "flk": "Bacteroidetes",
    "mgk": "Bacteroidetes",
    "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria",
    "cauj": "Alphaproteobacteria",
    "dri": "Alphaproteobacteria",
    "lgu": "Gammaproteobacteria",
    "lur": "Gammaproteobacteria",
    "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria",
    "pscf": "Gammaproteobacteria",
    "pzu": "Alphaproteobacteria",
    "racd": "Gammaproteobacteria",
    "tro": "Alphaproteobacteria",
    "vpe": "Betaproteobacteria",
}

# === Phyla-specific sorting ===
phyla_order = ['Alphaproteobacteria','Gammaproteobacteria', 'Actinobacteria', 'Bacteroidetes','Betaproteobacteria',  'Firmicutes']
model_order = []
for phylum in phyla_order:
    models_in_phylum = [model for model, p in phyla_dict.items() if p == phylum]
    sub_df = df[df['Models'].isin(models_in_phylum)]
    sorted_models = sub_df.sort_values(by='Rxns', ascending=False)['Models'].tolist()
    model_order.extend(sorted_models)

# === Reorder data ===
df = df.set_index('Models').loc[model_order]
models = df.index.tolist()
num_vars = len(models)

# === Extract metrics ===
rxns, draft_rxns = df['Rxns'].tolist(), df['DraftRxns'].tolist()
mets, draft_mets = df['Mets'].tolist(), df['DraftMets'].tolist()
genes, draft_genes = df['Genes'].tolist(), df['DraftGenes'].tolist()
n50_values, gf_values = df['Total Length (Mb)'].tolist(), df['TotalGenes'].tolist()
f1_score, mcc = df['F1'].tolist(), df['MCC'].tolist()
absolute_data = [n50_values, gf_values, f1_score, mcc]
labels = ['N50 (Mb)', 'GF', 'F1 Score', 'MCC']
def safe_norm(vals, pad_frac=0.03, pad_abs=0.0):
    vmin = min(vals)
    vmax = max(vals)
    if vmin == vmax:
        # avoid zero range
        vmin -= 1
        vmax += 1
    span = vmax - vmin
    return mpl.colors.Normalize(vmin=vmin - pad_frac*span - pad_abs,
                                vmax=vmax + pad_frac*span + pad_abs)

# === Normalizations (match absolute_data order) ===
normalizations = [
    safe_norm(n50_values, pad_frac=0.02),
    safe_norm(gf_values,  pad_frac=0.02),   # <-- uses Total Genes range
    safe_norm(f1_score,   pad_frac=0.00),   # typically [0,1]
    safe_norm(mcc,        pad_frac=0.00),   # typically [0,1]
]

# === Color maps (light to dark) ===
def ramp(name, colors):
    return LinearSegmentedColormap.from_list(name, colors)

# Monochromatic inner rings (distinct neutrals)
# 1) N50 (Mb)  -> Steel (blue-gray)
# 2) GF        -> Graphite (neutral gray)
# 3) F1 Score  -> Taupe (warm gray)
# 4) MCC       -> Slate (cool gray)

cmap_n50  = ramp('steel',   ['#F1F6FA', '#9BB6C6', '#2E4858'])
cmap_gf   = ramp('graphite',['#F4F4F4', '#BDBDBD', '#3F3F3F'])
cmap_f1   = ramp('taupe',   ['#F5F2EF', '#CBBEAF', '#6E5E4E'])
cmap_mcc  = ramp('slate',   ['#EFF2F3', '#A2B1B9', '#455A64'])

color_maps = [cmap_n50, cmap_gf, cmap_f1, cmap_mcc]
# === Phyla color assignment ===
phyla_color_map = {
    'Actinobacteria': '#1B9E77',
    'Bacteroidetes': '#D95F02',
    'Firmicutes': '#7570B3',
    'Alphaproteobacteria': '#E7298A',
    'Betaproteobacteria': '#66A61E',
    'Gammaproteobacteria': '#E6AB02'
}
phyla_colors = [phyla_color_map[phyla_dict[model]] for model in models]

# === Angular layout ===
group_sizes = []
for phylum in phyla_order:
    group_count = sum(1 for model in model_order if phyla_dict[model] == phylum)
    group_sizes.append(group_count)
# === ANGLES WITH GAPS BETWEEN PHYLA ===
gap_size = 0.25# radians of separation between phyla
angles = []
current_angle = 0

for size in group_sizes:
    for _ in range(size):
        angles.append(current_angle)
        current_angle += (2 * np.pi - len(group_sizes) * gap_size) / num_vars
    current_angle += gap_size  # add gap after group

angles += [angles[0]]  # close the circle

bar_width = (2 * np.pi - len(group_sizes) * gap_size) / num_vars / 3

# === Layout parameters ===
central_void_radius = 4
phyla_ring_height = 3
phyla_ring_offset = central_void_radius
ring_thickness = 4
ring_gap = 0.5
first_ring_bottom = phyla_ring_offset + phyla_ring_height + ring_gap
stacked_bar_scale = 0.008
stacked_bar_bottom = first_ring_bottom + len(absolute_data) * (ring_thickness + ring_gap)

# === Plot ===
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

# -- PHYLA RING
ax.bar(
    angles[:-1],
    [phyla_ring_height] * num_vars,
    width=2 * np.pi / num_vars,
    bottom=phyla_ring_offset,
    color=phyla_colors,
    edgecolor='white',
    linewidth=0.5
)

# -- METRIC RINGS
for i, (row, cmap, label) in enumerate(zip(absolute_data, color_maps, labels)):
    norm = normalizations[i]
    this_ring_bottom = first_ring_bottom + i * (ring_thickness + ring_gap)
    ax.bar(
        angles[:-1],
        [ring_thickness] * num_vars,
        width=2 * np.pi / num_vars,
        bottom=this_ring_bottom,
        color=cmap(norm(row)),
        edgecolor='white'
    )
    ax.text(
        0, this_ring_bottom + ring_thickness / 2,
        label, rotation=90, ha='center', va='center', size=11
    )

# -- STACKED BAR PLOTS (Rxns, Mets, Genes)
for i in range(num_vars):
    # Rxns
    angle = angles[i]
    ax.bar(angle - bar_width, draft_rxns[i] * stacked_bar_scale, width=bar_width,
           bottom=stacked_bar_bottom, color='cyan', edgecolor='white')
    ax.bar(angle - bar_width, (rxns[i] - draft_rxns[i]) * stacked_bar_scale, width=bar_width,
           bottom=draft_rxns[i] * stacked_bar_scale + stacked_bar_bottom, color='#32292F', edgecolor='white')

    # Mets
    ax.bar(angle, draft_mets[i] * stacked_bar_scale, width=bar_width,
           bottom=stacked_bar_bottom, color='yellowgreen', edgecolor='white')
    ax.bar(angle, (mets[i] - draft_mets[i]) * stacked_bar_scale, width=bar_width,
           bottom=draft_mets[i] * stacked_bar_scale + stacked_bar_bottom, color='#3F3047', edgecolor='white')

    # Genes
    ax.bar(angle + bar_width, draft_genes[i] * stacked_bar_scale, width=bar_width,
           bottom=stacked_bar_bottom, color='#3F3047', edgecolor='white')
    ax.bar(angle + bar_width, (genes[i] - draft_genes[i]) * stacked_bar_scale, width=bar_width,
           bottom=draft_genes[i] * stacked_bar_scale + stacked_bar_bottom, color='#3F3047', edgecolor='white')
# -- DOTTED REFERENCE CIRCLES (500 to 3500)
for ref in [500, 1500, 2500, 3500]:
    ref_height = ref * stacked_bar_scale + stacked_bar_bottom
    ax.plot(angles, [ref_height] * len(angles), '--', color='gray', alpha=0.5)
    ax.text(angles[0], ref_height, str(ref), color='gray', size=8, ha='left', va='bottom')

# -- Labels, aesthetics
ax.set_xticks(angles[:-1])
ax.set_xticklabels(models, size=8)
ax.set_yticklabels([])
ax.grid(False)
plt.title('Circular Heatmap with Donut Center and Stacked Bars', size=14)
plt.tight_layout()
plt.savefig('GEMstats_Final_Circular.pdf', format='pdf', bbox_inches='tight')
plt.show()

# Save the plot as PDF
# Add the legend for the phyla ring
# Set up the figure and axis for the circular stacked heatmap with requested color maps
# Set up the figure and axis for the circular stacked heatmap
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

# Plot heatmap rings and store colorbars
colorbars = []
ring_spacing = 5  # Spacing between rings
ring_height = 8   # Height of each ring
bottom = 7         # Starting position for the innermost ring
for i, (row, cmap, label) in enumerate(zip(absolute_data, color_maps, labels)):
    norm = normalizations[i]
    values = list(row) + [row[0]]  # Close the circle by repeating the first value
    bars = ax.bar(
        angles[:-1],
        np.ones(len(angles[:-1])) * ring_height,
        width=2 * np.pi / num_vars,
        bottom=i * ring_spacing + bottom,
        color=cmap(norm(values)),
        edgecolor='white'
    )
    # Create a colorbar for this ring
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    colorbars.append(sm)

# Add the reordered model labels to the outer edge
ax.set_xticks(angles[:-1])
ax.set_xticklabels(models)

# Add legends as colorbars for heatmap rings
cbar_positions = [0.8, 0.65, 0.5, 0.35]  # Vertical positions for colorbars
for sm, label, pos in zip(colorbars, labels, cbar_positions):
    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation='horizontal',
        fraction=0.02,
        pad=0.1,
        aspect=40
    )
    cbar.ax.set_title(label, size=10)
    cbar.ax.tick_params(labelsize=8)
    cbar.ax.set_position([0.75, pos, 0.2, 0.02])  # Adjust colorbar position

# Add a legend for the phyla ring
phyla_patches = [
    mpl.patches.Patch(color=color, label=phylum)
    for phylum, color in phyla_color_map.items()
]
phyla_legend = ax.legend(handles=phyla_patches, loc='upper left', bbox_to_anchor=(1.1, 1.05), title="Phyla")

# Add legends for the stacked bar plots
stacked_bar_patches = [
    mpl.patches.Patch(color='cyan', label='DraftRxns'),
    mpl.patches.Patch(color='#32292F', label='Rxns (Complete - Draft)'),
    mpl.patches.Patch(color='yellowgreen', label='DraftMets'),
    mpl.patches.Patch(color='#3F3047', label='Mets/Genes (Complete - Draft)'),
]
stacked_bar_legend = ax.legend(handles=stacked_bar_patches, loc='upper left', bbox_to_anchor=(1.1, 0.25), title="Stacked Bars")

# Add legends to the plot
fig.add_artist(phyla_legend)
fig.add_artist(stacked_bar_legend)

# Hide radial ticks and grid for the heatmap
ax.grid(False)
ax.set_yticklabels([])

# Set the title
plt.title('Circular Stacked Heatmap with Central Void and Legends', size=16)
plt.savefig('GEMstatsFinal_with_heatmap_legends.pdf', format='pdf', bbox_inches='tight')

# Show the updated plot
plt.show()

# Save the updated plot as PDF
# Set up the figure and axis for the circular stacked heatmap with requested color maps and legends
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

# Plot heatmap rings
for i, (row, cmap, label) in enumerate(zip(absolute_data, color_maps, labels)):
    norm = mpl.colors.Normalize(vmin=min(row) - 0.1 * abs(min(row)), vmax=max(row) + 0.1 * abs(max(row)))
    values = list(row) + [row[0]]  # Close the circle by repeating the first value
    ax.bar(angles[:-1], np.ones(len(angles[:-1])), width=2 * np.pi / num_vars, bottom=i + bottom, color=cmap(norm(values)), edgecolor='white')
    ax.text(0, i + bottom + 1, label, rotation=90, size=12, ha='center', va='center')  # Add label for each ring

# Add the reordered model labels to the outer edge
ax.set_xticks(angles[:-1])
ax.set_xticklabels(models)

# Stacked bar plots without displaying numbers on the plot
for i in range(num_vars):
    angle = angles[i]  # get the correct angle with gap-aware logic
    # now use 'angle' as usual below
    # Rxns (complete minus draft) and DraftRxns
    ax.bar(angle - bar_width, draft_rxns[i] * stacked_bar_scale, width=bar_width, bottom=bottom + len(absolute_data), color='cyan', edgecolor='white', label='DraftRxns' if i == 0 else "")
    ax.bar(angle - bar_width, (rxns[i] - draft_rxns[i]) * stacked_bar_scale, width=bar_width, bottom=draft_rxns[i] * stacked_bar_scale + bottom + len(absolute_data), color='blue', edgecolor='white', label='Rxns' if i == 0 else "")
    
    # Mets (complete minus draft) and DraftMets
    ax.bar(angle, draft_mets[i] * stacked_bar_scale, width=bar_width, bottom=bottom + len(absolute_data), color='yellowgreen', edgecolor='white', label='DraftMets' if i == 0 else "")
    ax.bar(angle, (mets[i] - draft_mets[i]) * stacked_bar_scale, width=bar_width, bottom=draft_mets[i] * stacked_bar_scale + bottom + len(absolute_data), color='green', edgecolor='white', label='Mets' if i == 0 else "")
    
    # Genes (complete minus draft) and DraftGenes
    ax.bar(angle + bar_width, draft_genes[i] * stacked_bar_scale, width=bar_width, bottom=bottom + len(absolute_data), color='orange', edgecolor='white', label='DraftGenes' if i == 0 else "")
    ax.bar(angle + bar_width, (genes[i] - draft_genes[i]) * stacked_bar_scale, width=bar_width, bottom=draft_genes[i] * stacked_bar_scale + bottom + len(absolute_data), color='red', edgecolor='white', label='Genes' if i == 0 else "")

# Add dotted reference circles from 500 to 1500, 2500, and so on
for ref in [500, 1500, 2500, 3500, 4500, 5500]:
    ax.plot(angles, [ref * stacked_bar_scale + bottom + len(absolute_data)] * len(angles), '--', color='gray', alpha=0.5)

# Hide radial ticks and grid for the heatmap
ax.grid(False)
ax.set_yticklabels([])

# Set the title
plt.title('Circular Stacked Heatmap with Central Void and Legends', size=16)

# Add the legends for heatmap and bar colors
handles = [
    mpl.patches.Patch(color='cyan', label='DraftRxns'),
    mpl.patches.Patch(color='blue', label='Rxns'),
    mpl.patches.Patch(color='yellowgreen', label='DraftMets'),
    mpl.patches.Patch(color='green', label='Mets'),
    mpl.patches.Patch(color='orange', label='DraftGenes'),
    mpl.patches.Patch(color='red', label='Genes')
]

# Legend for the bar plots
plt.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.1, 1.1))

# Colorbar legends for the heatmap rings
cmap1 = mpl.cm.ScalarMappable(cmap=plt.cm.Oranges, norm=safe_norm(n50_values, pad_frac=0.02))
cmap2 = mpl.cm.ScalarMappable(cmap=plt.cm.Purples, norm=safe_norm(gf_values, pad_frac=0.02))
cmap5 = mpl.cm.ScalarMappable(cmap=plt.cm.Reds, norm=safe_norm(f1_score, pad_frac=0.00))
cmap6 = mpl.cm.ScalarMappable(cmap=plt.cm.Greys, norm=safe_norm(mcc, pad_frac=0.00))

# Create colorbars for the heatmap rings
cbar1 = plt.colorbar(cmap1, ax=ax, orientation='vertical', pad=0.1, fraction=0.03)
cbar1.set_label('N50 (Mb)')

cbar2 = plt.colorbar(cmap2, ax=ax, orientation='vertical', pad=0.1, fraction=0.03)
cbar2.set_label('GF')

cbar5 = plt.colorbar(cmap5, ax=ax, orientation='vertical', pad=0.1, fraction=0.03)
cbar5.set_label('F1 Score')

cbar6 = plt.colorbar(cmap6, ax=ax, orientation='vertical', pad=0.1, fraction=0.03)
cbar6.set_label('MCC')
# Add legend for phyla ring
for phylum, color in phyla_color_map.items():
    ax.bar(0, 0, color=color, label=phylum)  # Add invisible bars for the legend
ax.legend(loc="upper right", bbox_to_anchor=(1.1, 1.1), title="Phyla")
# Show the plot
plt.show()

# Save the plot as PDF
plt.savefig('GEMsFinalWithLegends.pdf', format='pdf', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib

# === Load Data ===
file_path = './GEMStats.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet2')

# === Define phyla mapping ===
phyla_dict = {
    "cohm": "Firmicutes",
    "cuvk": "Actinobacteria",
    "cxy": "Actinobacteria",
    "mip": "Actinobacteria",
    "rfa": "Actinobacteria",
    "cpi": "Bacteroidetes",
    "flk": "Bacteroidetes",
    "mgk": "Bacteroidetes",
    "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria",
    "cauj": "Alphaproteobacteria",
    "dri": "Alphaproteobacteria",
    "lgu": "Gammaproteobacteria",
    "lur": "Gammaproteobacteria",
    "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria",
    "pscf": "Gammaproteobacteria",
    "pzu": "Alphaproteobacteria",
    "racd": "Gammaproteobacteria",
    "tro": "Alphaproteobacteria",
    "vpe": "Betaproteobacteria",
}

# === Phyla-specific sorting ===
phyla_order = [
    'Alphaproteobacteria',
    'Gammaproteobacteria',
    'Actinobacteria',
    'Bacteroidetes',
    'Betaproteobacteria',
    'Firmicutes'
]

model_order = []
for phylum in phyla_order:
    models_in_phylum = [model for model, p in phyla_dict.items() if p == phylum]
    sub_df = df[df['Models'].isin(models_in_phylum)]
    sorted_models = sub_df.sort_values(by='Rxns', ascending=False)['Models'].tolist()
    model_order.extend(sorted_models)

# === Reorder data ===
df = df.set_index('Models').loc[model_order]
models = df.index.tolist()
num_vars = len(models)

# === Extract metrics ===
rxns, draft_rxns = df['Rxns'].tolist(), df['DraftRxns'].tolist()
mets, draft_mets = df['Mets'].tolist(), df['DraftMets'].tolist()
genes, draft_genes = df['Genes'].tolist(), df['Genes'].tolist()

# (You still extract these in case you need them later, but they are not plotted now)
n50_values, gf_values = df['Total Length (Mb)'].tolist(), df['TotalGenes'].tolist()
f1_score, mcc = df['F1'].tolist(), df['MCC'].tolist()

# === Phyla color assignment ===
phyla_color_map = {
    'Actinobacteria': '#1B9E77',
    'Bacteroidetes': '#D95F02',
    'Firmicutes': '#7570B3',
    'Alphaproteobacteria': '#E7298A',
    'Betaproteobacteria': '#66A61E',
    'Gammaproteobacteria': '#E6AB02'
}
phyla_colors = [phyla_color_map[phyla_dict[model]] for model in models]

# === Angular layout with gaps between phyla ===
group_sizes = []
for phylum in phyla_order:
    group_count = sum(1 for model in model_order if phyla_dict[model] == phylum)
    group_sizes.append(group_count)

gap_size = 0.25  # radians of separation between phyla
angles = []
current_angle = 0

for size in group_sizes:
    for _ in range(size):
        angles.append(current_angle)
        current_angle += (2 * np.pi - len(group_sizes) * gap_size) / num_vars
    current_angle += gap_size  # add gap after group

angles += [angles[0]]  # close the circle

bar_width = (2 * np.pi - len(group_sizes) * gap_size) / num_vars / 3

# === Layout parameters (NO central void) ===
phyla_ring_height = 3
phyla_ring_offset = 0.0       # start at center => no inner "donut" hole
ring_gap = 0.5
stacked_bar_scale = 0.004
stacked_bar_bottom = phyla_ring_offset + phyla_ring_height + ring_gap

# === Plot: only phyla ring + stacked bars ===
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

# -- PHYLA RING
ax.bar(
    angles[:-1],
    [phyla_ring_height] * num_vars,
    width=2 * np.pi / num_vars,
    bottom=phyla_ring_offset,
    color=phyla_colors,
    edgecolor='white',
    linewidth=0.5
)

# -- STACKED BAR PLOTS (Rxns, Mets, Genes)
for i in range(num_vars):
    angle = angles[i]

    # Rxns
    ax.bar(
        angle - bar_width,
        draft_rxns[i] * stacked_bar_scale,
        width=bar_width,
        bottom=stacked_bar_bottom,
        color='cyan',
        edgecolor='white'
    )
    ax.bar(
        angle - bar_width,
        (rxns[i] - draft_rxns[i]) * stacked_bar_scale,
        width=bar_width,
        bottom=draft_rxns[i] * stacked_bar_scale + stacked_bar_bottom,
        color='#32292F',
        edgecolor='white'
    )

    # Mets
    ax.bar(
        angle,
        draft_mets[i] * stacked_bar_scale,
        width=bar_width,
        bottom=stacked_bar_bottom,
        color='yellowgreen',
        edgecolor='white'
    )
    ax.bar(
        angle,
        (mets[i] - draft_mets[i]) * stacked_bar_scale,
        width=bar_width,
        bottom=draft_mets[i] * stacked_bar_scale + stacked_bar_bottom,
        color='#3F3047',
        edgecolor='white'
    )

    # Genes
    ax.bar(
        angle + bar_width,
        draft_genes[i] * stacked_bar_scale,
        width=bar_width,
        bottom=stacked_bar_bottom,
        color='#3F3047',
        edgecolor='white'
    )
    ax.bar(
        angle + bar_width,
        (genes[i] - draft_genes[i]) * stacked_bar_scale,
        width=bar_width,
        bottom=draft_genes[i] * stacked_bar_scale + stacked_bar_bottom,
        color='#3F3047',
        edgecolor='white'
    )

# -- DOTTED REFERENCE CIRCLES (500 to 3500)
for ref in [500, 1500, 2500, 3500]:
    ref_height = ref * stacked_bar_scale + stacked_bar_bottom
    ax.plot(angles, [ref_height] * len(angles), '--', color='gray', alpha=0.5)
    ax.text(
        angles[0],
        ref_height,
        str(ref),
        color='gray',
        size=8,
        ha='left',
        va='bottom'
    )

# -- Labels, aesthetics
ax.set_xticks(angles[:-1])
ax.set_xticklabels(models, size=8)
ax.set_yticklabels([])
ax.grid(False)

plt.title('Phylogenetic Ring with Stacked Rxns/Mets/Genes', size=14)
plt.tight_layout()
plt.savefig('GEMstats_phylo_stackedbars.pdf', format='pdf', bbox_inches='tight')
plt.show()
